# 8. LLM Agent Layer (판단 계층 설계)

## 이번 노트북에서 할 것
- 여러 toxicophore 문제 중 "어떤 것부터 고칠지" LLM이 판단하도록 설계
- 여러 치환 후보(candidates) 중 "어떤 게 이 상황에 맞는지" LLM이 선택하도록 설계
- 도구 출력(JSON) → LLM 프롬프트 구성 → LLM 응답 파싱 → propose_fix() 호출까지 잇는 구조 시험
- 지금까지 고정해뒀던 `problems[0]`, `candidate_idx=0` 자리를 LLM 판단으로 교체

## 간략한 정리 (07까지)
- 도구 계층 완전히 완성 + 검증됨: data_prep / baseline classifier(AUROC 0.821) /
  toxicophore_detector / replacement_library(7개 규칙) / molecule_editor
  (find_core_and_target, reassemble_molecule, propose_fix, canonicalize,
  iterative_fix_loop)
- iterative_fix_loop 검증 완료: alkyl_halide 케이스 success, nitro_group 케이스
  부분성공+no_known_fix로 정상 종료 확인
- 현재 루프는 "문제가 여러 개면 무조건 첫 번째부터", "후보가 여러 개면 무조건
  0번째부터" 고치는 규칙 기반 구조 → 이게 오늘 LLM으로 교체할 부분
- docs/limitations.md: 대사/약물상호작용은 스코프 밖임을 명시해둠

## 다음에 해야 할 것 (오늘 끝나면)
- LLM 판단까지 붙은 최종 루프를 held-out set 일부에 돌려서 성공률/개선율 통계
- 실제 사례(개발 중단된 약물 등) 케이스스터디 1~2개 준비
- 제안서(hwpx) 초안 작성 시작 (마감 8/7)

In [18]:
# 셀 1
!pip install rdkit -q
!pip install -U google-genai -q

In [19]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 87 (delta 27), reused 59 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 264.08 KiB | 6.95 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/laidd-2026
/content/laidd-2026


In [20]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [21]:
# 셀 4
from rdkit import Chem
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop

data = load_tox21_clean()
print("도구 로드 확인 완료")

[06:00:33] WARNING: not removing hydrogen atom without neighbors
[06:00:33] Explicit valence for atom # 8 Al, 6, is greater than permitted
[06:00:33] Explicit valence for atom # 3 Al, 6, is greater than permitted
[06:00:33] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:00:33] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:00:33] Explicit valence for atom # 9 Al, 6, is greater than permitted
[06:00:33] Explicit valence for atom # 5 Al, 6, is greater than permitted
[06:00:34] Explicit valence for atom # 16 Al, 6, is greater than permitted
[06:00:34] Explicit valence for atom # 20 Al, 6, is greater than permitted
[06:00:34] WARNING: not removing hydrogen atom without neighbors


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개
도구 로드 확인 완료


In [22]:
# 셀 5
from google import genai

gemini_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_key)
print("Gemini 클라이언트 준비 완료")

Gemini 클라이언트 준비 완료


In [23]:
response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents='안녕, 잘 연결됐는지 한 문장으로 답해줘.'
)
print(response.text)

안녕하세요, 연결이 아주 잘 되었으니 무엇이든 편하게 말씀해 주세요.


In [24]:
test_mol_smiles = None
for s in data['smiles_train'][:500]:
    result = detect_toxicophores(s)
    if any(r['rule_name'] == 'nitro_group' for r in result):
        test_mol_smiles = s
        break

problems = detect_toxicophores(test_mol_smiles)
print("테스트 분자:", test_mol_smiles)
print("현재 문제들:", problems)

테스트 분자: O=[N+]([O-])c1ccc(C=NO)o1
현재 문제들: [{'rule_name': 'imine_1', 'atom_indices': [7, 8]}, {'rule_name': 'nitro_group', 'atom_indices': [0, 1, 2]}, {'rule_name': 'oxime_1', 'atom_indices': [7, 8, 9]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [1, 2]}]


In [25]:
import json

def ask_llm_which_problem_to_fix(smiles, problems):
    """여러 문제 중 어떤 것부터 고칠지 LLM에게 판단을 요청."""
    # 우리가 실제로 치환 지식을 가진 문제만 후보로 제시
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None  # 애초에 고칠 수 있는 게 없으면 LLM 호출도 불필요

    if len(known) == 1:
        return known[0]['rule_name']  # 하나뿐이면 판단 자체가 불필요

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    response = client.models.generate_content(
        model='gemini-3.5-flash',
        contents=prompt
    )
    return response.text


result = ask_llm_which_problem_to_fix(test_mol_smiles, problems)
print(result)

nitro_group


In [26]:
multi_known_test = None
for s in data['smiles_train'][:2000]:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 2:
        multi_known_test = s
        break

print("찾은 분자:", multi_known_test)
if multi_known_test:
    print("문제들:", detect_toxicophores(multi_known_test))

찾은 분자: Nc1ccc(NCCO)c([N+](=O)[O-])c1
문제들: [{'rule_name': 'aniline', 'atom_indices': [0, 1, 2, 3, 4, 9, 13]}, {'rule_name': 'nitro_group', 'atom_indices': [10, 11, 12]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [10, 12]}]


In [27]:
problems2 = detect_toxicophores(multi_known_test)
result2 = ask_llm_which_problem_to_fix(multi_known_test, problems2)
print(result2)

{"rule_name": "nitro_group", "reason": "니트로기는 유전독성(Ames mutagenicity) 및 대사적 불안정성을 유발하는 가장 대표적인 독성 작용기이므로 최우선적으로 치환해야 합니다."}


In [28]:
import json

def ask_llm_which_problem_to_fix(smiles, problems):
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None
    if len(known) == 1:
        return {"rule_name": known[0]['rule_name'], "reason": "유일한 치환 가능 후보"}

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    response = client.models.generate_content(model='gemini-3.5-flash', contents=prompt)
    text = response.text.strip()

    # 혹시 코드펜스(```json ... ```)가 섞여 나올 경우 대비해 제거
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]

    try:
        parsed = json.loads(text)
        return parsed
    except json.JSONDecodeError:
        return {"rule_name": known[0]['rule_name'], "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}


# 테스트
decision = ask_llm_which_problem_to_fix(multi_known_test, problems2)
print(decision)
print("타입:", type(decision))

{'rule_name': 'nitro_group', 'reason': '니트로기는 체내 환원 대사를 통해 강력한 유전독성(Ames positive) 및 대사 불안정성을 유발하는 대표적인 고위험 작용기이므로 아닐린보다 우선적으로 치환해야 합니다.'}
타입: <class 'dict'>


In [29]:
%%writefile src/tools/agent.py
import json
from src.tools.replacement_library import get_replacement_candidates


def ask_llm_which_problem_to_fix(client, model_name, smiles, problems):
    """여러 toxicophore 중 어떤 것부터 고칠지 LLM에게 판단을 요청.
    client: google.genai.Client 인스턴스
    model_name: 사용할 모델 이름 (예: 'gemini-3.5-flash')
    """
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None
    if len(known) == 1:
        return {"rule_name": known[0]['rule_name'], "reason": "유일한 치환 가능 후보"}

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    response = client.models.generate_content(model=model_name, contents=prompt)
    text = response.text.strip()

    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"rule_name": known[0]['rule_name'], "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}

Overwriting src/tools/agent.py


In [30]:
def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None):
    """진단->치환->재평가를 반복. llm_client가 주어지면 여러 문제 중
    어떤 걸 고칠지 LLM이 판단, 없으면 기존처럼 리스트 순서대로 처리."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        # 여기가 핵심 변경 지점
        if llm_client is not None:
            decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems)
            target_rule = decision['rule_name']
            decision_reason = decision.get('reason', '')
        else:
            target_rule = known_problems[0]['rule_name']
            decision_reason = "규칙 기반(리스트 순서대로)"

        fixed = propose_fix(current, target_rule, candidate_idx)

        if fixed is None or not fixed['is_valid']:
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "decision_reason": decision_reason,
            "candidate_used": fixed['candidate_used'],
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

In [31]:
%%writefile src/tools/agent.py
import json
from src.tools.replacement_library import get_replacement_candidates


def ask_llm_which_problem_to_fix(client, model_name, smiles, problems):
    """여러 toxicophore 중 어떤 것부터 고칠지 LLM에게 판단을 요청."""
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None
    if len(known) == 1:
        return {"rule_name": known[0]['rule_name'], "reason": "유일한 치환 가능 후보"}

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    response = client.models.generate_content(model=model_name, contents=prompt)
    text = response.text.strip()

    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"rule_name": known[0]['rule_name'], "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}

Overwriting src/tools/agent.py


In [32]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def find_core_and_target(smiles: str, rule_name: str):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)

    best_match = None
    for core, chain in fragments:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            part_mol = Chem.MolFromSmiles(part.replace('[*:1]', '[H]'))
            if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
                continue
            frag_heavy_atoms = part_mol.GetNumHeavyAtoms()
            if frag_heavy_atoms == pattern_size:
                return {"core": parts[1 - i], "target_removed": part}
            if best_match is None:
                best_match = {"core": parts[1 - i], "target_removed": part}
    return best_match


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None):
    """진단->치환->재평가를 반복. llm_client가 주어지면 여러 문제 중
    어떤 걸 고칠지 LLM이 판단, 없으면 리스트 순서(규칙 기반)로 처리."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        if llm_client is not None:
            decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems)
            target_rule = decision['rule_name']
            decision_reason = decision.get('reason', '')
        else:
            target_rule = known_problems[0]['rule_name']
            decision_reason = "규칙 기반(리스트 순서대로)"

        fixed = propose_fix(current, target_rule, candidate_idx)

        if fixed is None or not fixed['is_valid']:
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "decision_reason": decision_reason,
            "candidate_used": fixed['candidate_used'],
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

Overwriting src/tools/molecule_editor.py


In [34]:
import importlib
import src.tools.agent
import src.tools.molecule_editor
importlib.reload(src.tools.agent)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop

# 규칙 기반 (기존 방식)
result_rule = iterative_fix_loop(multi_known_test, max_iterations=10)
print("=== 규칙 기반 ===")
print("상태:", result_rule['status'])
for h in result_rule['history']:
    print(h.get('fixed_rule'), '|', h.get('decision_reason'))

# LLM 기반 (새 방식)
result_llm = iterative_fix_loop(multi_known_test, max_iterations=10, llm_client=client, llm_model='gemini-3.5-flash')
print("\n=== LLM 기반 ===")
print("상태:", result_llm['status'])
for h in result_llm['history']:
    print(h.get('fixed_rule'), '|', h.get('decision_reason'))

=== 규칙 기반 ===
상태: success
None | None
aniline | 규칙 기반(리스트 순서대로)

=== LLM 기반 ===
상태: success
None | None
nitro_group | 니트로기는 생체 내에서 쉽게 환원되어 유전독성 및 세포독성을 유발하는 반응성 대사체를 형성하는 대표적인 고위험 독성 작용기이므로 가장 우선적으로 치환되어야 합니다.
aniline | 유일한 치환 가능 후보


In [17]:
!git add src/tools/agent.py src/tools/molecule_editor.py
!git commit -m "Add LLM-based decision layer (agent.py) for choosing which toxicophore to fix first; compare against rule-based ordering"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 525d75d] Add LLM-based decision layer (agent.py) for choosing which toxicophore to fix first; compare against rule-based ordering
 2 files changed, 54 insertions(+), 10 deletions(-)
 create mode 100644 src/tools/agent.py
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 1.78 KiB | 1.78 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   6c03ed9..525d75d  main -> main
